In [2]:
import pandas as pd
import joblib

# LOADING AND PREPROCESSING NEW DATA
print("Loading new application data...")
new_applicants = pd.read_excel('processed_district_data_with_property_ids.xlsx')

# Identifying direct income/threshold columns to drop
income_cols = [col for col in new_applicants.columns if col.startswith('in') and any(char.isdigit() for char in col) or col.startswith('incometaxthreshold_')]

# Metadata columns to drop (Including IDs and text columns)
metadata_cols = ['hasfamilyid','Property_ID', 'BPL_Target', 'district', 'blocktown', 'wardvillage', 'r_u', 'familyRange']

# Noisy proxy columns to drop 
noisy_cols = ['is_Child', 'is_Housewife', 'is_Senior Citizen', 'is_Student', 'is_Farmer', 'is_Labour', 'is_Pensioner/Retired']

cols_to_drop = metadata_cols + noisy_cols + income_cols 

# Creating a clean dataframe (X_new) that ONLY contains the training features
X_new = new_applicants.drop(columns=[col for col in cols_to_drop if col in new_applicants.columns])

# Failsafe: Ensuring only numeric columns remain
X_new = X_new.select_dtypes(include=['int64', 'float64'])

print(f"Data preprocessed. Features ready for model: {X_new.shape[1]}")

# LOADING THE SAVED BRAIN & PREDICT
print("Loading trained model...")

# Change this joblib file name to the one which needs to be used from the 2nd Classification's Model Comparison folder or New Model Comparison folder and move that file in the current folder.
model = joblib.load('2nd_pipeline.joblib') 

# The model outputs a 1 (BPL) or a 0 (Non-BPL) based purely on the 60 proxy features
print("Generating predictions...")
predictions = model.predict(X_new)

# Getting the model's confidence score (Probability of being BPL: 0.0 to 1.0)
confidence_scores = model.predict_proba(X_new)[:, 1]

# EXPORTING THE RESULTS FOR ALL FAMILIES
# Attaching predictions back to the ORIGINAL dataframe so no IDs or metadata are lost
new_applicants['Predicted_Status'] = predictions
new_applicants['BPL_Probability_Score'] = confidence_scores * 100 

# Saving the full list of ALL families (APL and BPL) with their AI evaluations
output_filename = '2nd_families_evaluated.xlsx'
new_applicants.to_excel(output_filename, index=False)

print(f"\n✅ Success! Scanned {len(new_applicants)} applications.")
print(f"Exported all predictions and confidence scores to '{output_filename}'.")

Loading new application data...
Data preprocessed. Features ready for model: 29
Loading trained model...
Generating predictions...

✅ Success! Scanned 2301 applications.
Exported all predictions and confidence scores to '2nd_families_evaluated.xlsx'.
